# Task 3: Constraint Satisfaction Problems (Kakuro)
**Môn học:** Introduction to Artificial Intelligence (503043) - Trường Đại học Tôn Đức Thắng (TDTU)  
**Thành viên thực hiện:** Nhóm sinh viên  

## Mục lục
1. **Mô hình hóa bài toán CSP: Biến, Miền giá trị, và Ràng buộc Run**
2. **Kiểm tra tính nhất quán (Tổng Clue Ngang == Tổng Clue Dọc)**
3. **Giải bài toán 9x9 với Backtracking + AC-3 + Heuristics (MRV, LCV)**
4. **Đánh giá Correctness, Stability, và Effectiveness (Ablation Study)**

### 1. Nạp và Khởi tạo bài toán CSP từ dữ liệu JSON

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from task3_kakuro.csp.csp_problem import KakuroCSP

csp = KakuroCSP()
csp.load_from_json('task3_kakuro/puzzles/complex_9x9.json')
print(f"[*] Đã nạp thành công câu đố: {csp.name}")
print(f"[*] Kích thước bảng: {csp.rows}x{csp.cols}")
print(f"[*] Số biến ô trắng (Variables): {len(csp.variables)}")
print(f"[*] Số lượng ràng buộc Run: {len(csp.runs)} (14 ngang, 14 dọc)")
h_sum, v_sum = csp.get_clue_sums()
print(f"[*] Kiểm tra tính toàn vẹn Clue (Tổng H == Tổng V): {csp.validate_puzzle_integrity()} (H={h_sum}, V={v_sum})")

[*] Đã nạp thành công câu đố: Kakuro Complex 9x9 (Problem Page 5 Test Case)
[*] Kích thước bảng: 9x9
[*] Số biến ô trắng (Variables): 48
[*] Số lượng ràng buộc Run: 28 (14 ngang, 14 dọc)
[*] Kiểm tra tính toàn vẹn Clue (Tổng H == Tổng V): True (H=249, V=249)


### 2. Giải bài toán phức tạp 9x9 và Truy vết hoạt động (Solver Tracer)

In [2]:
from task3_kakuro.solver.backtracking_solver import KakuroBacktrackingSolver
from task3_kakuro.solver.tracer import SolverTracer

tracer = SolverTracer()
solver = KakuroBacktrackingSolver(use_ac3=True, use_mrv=True, use_lcv=True, tracer=tracer)

print("[*] Bắt đầu giải bài toán Kakuro 9x9...")
result = solver.solve(csp)

print("-" * 40)
print(" KẾT QUẢ GIẢI (SOLVE RESULT)")
print("-" * 40)
print(f" - Đã giải thành công    : {result.is_solved}")
print(f" - Thời gian thực thi    : {result.execution_time:.4f} giây")
print(f" - Số nút mở rộng (nodes): {result.nodes_explored}")
print(f" - Số lần quay lui       : {result.backtracks}")
print(f" - Tổng số sự kiện tracer: {len(tracer.events)}")

if result.is_solved:
    print("\n[*] Ma trận nghiệm tìm được:")
    for r in range(csp.rows):
        row_str = []
        for c in range(csp.cols):
            pos = (r, c)
            if pos in csp.variables:
                row_str.append(f" {result.solution_grid[pos]} ")
            else:
                row_str.append(" # ")
        print("  " + "".join(row_str))
    print("\n[+] Tất cả các ràng buộc Clue đều được thỏa mãn 100%!")

[*] Bắt đầu giải bài toán Kakuro 9x9...
----------------------------------------
 KẾT QUẢ GIẢI (SOLVE RESULT)
----------------------------------------
 - Đã giải thành công    : True
 - Thời gian thực thi    : 0.1697 giây
 - Số nút mở rộng (nodes): 11383
 - Số lần quay lui       : 11335
 - Tổng số sự kiện tracer: 31246

[*] Ma trận nghiệm tìm được:
   #  #  #  #  #  #  #  #  # 
   #  #  5  9  8  #  1  2  # 
   #  #  2  7  1  3  4  6  5 
   #  8  6  #  #  1  2  8  9 
   #  9  8  #  3  2  8  9  # 
   #  #  9  8  6  5  #  7  5 
   #  3  4  5  2  #  #  4  2 
   #  9  3  7  1  6  8  5  # 
   #  #  7  9  #  1  4  3  # 

[+] Tất cả các ràng buộc Clue đều được thỏa mãn 100%!


### 3. Đánh giá Correctness, Stability, và Effectiveness (Ablation Study)

In [3]:
from task3_kakuro.experiments.evaluate_kakuro import KakuroEvaluator

evaluator = KakuroEvaluator(puzzle_path='task3_kakuro/puzzles/complex_9x9.json')
evaluator.run_all_evaluations()

     ĐÁNH GIÁ HIỆU NĂNG & ĐỘ ỔN ĐỊNH SOLVER KAKURO CSP

[*] Kiểm thử tính ổn định (Stability Test) qua 5 lần chạy độc lập:


  - Lần 1: Thời gian=0.1443s, Nodes=11383, Backtracks=11335


  - Lần 2: Thời gian=0.1448s, Nodes=11383, Backtracks=11335


  - Lần 3: Thời gian=0.1464s, Nodes=11383, Backtracks=11335


  - Lần 4: Thời gian=0.1437s, Nodes=11383, Backtracks=11335


  - Lần 5: Thời gian=0.1446s, Nodes=11383, Backtracks=11335
[+] Tính đúng đắn (Independent Verification): HỢP LỆ 100%
[+] Tính nhất quán của nghiệm: ĐỒNG NHẤT 100%
[+] Thời gian trung bình: 0.1448 giây

[*] Nghiên cứu cắt giảm (Ablation Study) - So sánh hiệu quả các cấu hình:


  * Pure Backtracking (No Heuristics, Arbitrary): Timeout (>3.0s) | Nodes: >28,086 | Backtracks: >28,086


  * Backtracking + MRV: 0.3228s | Nodes: 35,075 | Backtracks: 35,027
  * Backtracking + MRV + LCV: 0.1408s | Nodes: 11,383 | Backtracks: 11,335


  * Backtracking + AC-3 + MRV + LCV (Full): 0.1463s | Nodes: 11,383 | Backtracks: 11,335
